# HE Inference Demo

This notebook is the canonical homomorphic-encryption inference demo for the 6D PaySim baseline. It performs **encrypted inference only** and does not retrain models or regenerate baseline artifacts.

We compare plaintext and CKKS encrypted inference for four exported models:

- Ridge score baseline
- Logistic Regression raw logit
- LinearSVC decision function
- Degree-2 polynomial Logistic Regression raw score


## 1. Project Overview

Training stays outside HE. This notebook assumes the baseline notebook has already exported:

- model parameters and model pickles
- feature order metadata
- scaler statistics
- an HE evaluation subset
- plaintext reference scores

The main goal is to compare plaintext and encrypted inference outputs in a clean, reproducible way.


In [1]:
from __future__ import annotations

import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Ensure project root is on path so `src` is importable
sys.path.insert(0, str(Path(".").resolve()))

from src.he_utils import (
    CKKS_PARAMETER_OPTIONS,
    TENSEAL_AVAILABLE,
    TENSEAL_IMPORT_ERROR,
    build_ckks_context,
    encrypted_linear_score,
    run_ckks_batch_demo,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

## 2. Imports And Configuration

We prefer TenSEAL with CKKS when available. If TenSEAL is missing, the notebook still performs all plaintext consistency checks and leaves the HE experiment cells ready to rerun later.


In [2]:
PROJECT_ROOT = Path(".").resolve()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

FEATURE_ORDER_PATH = ARTIFACT_DIR / "feature_order_6d.json"
SCALER_PATH = ARTIFACT_DIR / "baseline_6d_scaler.pkl"
RIDGE_PATH = ARTIFACT_DIR / "baseline_6d_ridge.pkl"
LOGREG_PATH = ARTIFACT_DIR / "baseline_6d_logreg.pkl"
LINEAR_SVC_PATH = ARTIFACT_DIR / "linear_svc_model.pkl"
POLY2_LOGREG_PATH = ARTIFACT_DIR / "poly2_logreg_model.pkl"
POLY2_TRANSFORMER_PATH = ARTIFACT_DIR / "poly2_transformer.pkl"
BASELINE_COMPARISON_PATH = ARTIFACT_DIR / "baseline_model_comparison_summary.json"
HE_FEATURES_PATH = ARTIFACT_DIR / "he_eval_scaled_features_6d.csv"
HE_META_PATH = ARTIFACT_DIR / "he_eval_meta_6d.csv"
HE_SCORES_PATH = ARTIFACT_DIR / "he_eval_plaintext_scores_6d.csv"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

artifact_paths = {
    "feature_order": FEATURE_ORDER_PATH,
    "scaler": SCALER_PATH,
    "ridge": RIDGE_PATH,
    "logreg": LOGREG_PATH,
    "linear_svc": LINEAR_SVC_PATH,
    "poly2_logreg": POLY2_LOGREG_PATH,
    "poly2_transformer": POLY2_TRANSFORMER_PATH,
    "baseline_comparison_summary": BASELINE_COMPARISON_PATH,
    "he_features": HE_FEATURES_PATH,
    "he_meta": HE_META_PATH,
    "he_scores": HE_SCORES_PATH,
}

missing_paths = [str(path) for path in artifact_paths.values() if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(missing_paths))

print("TenSEAL available:", TENSEAL_AVAILABLE)
if not TENSEAL_AVAILABLE:
    print("TenSEAL import error:", repr(TENSEAL_IMPORT_ERROR))

display(pd.DataFrame({"artifact": artifact_paths.keys(), "path": [str(path) for path in artifact_paths.values()]}))


TenSEAL available: True


,artifact,path
0,feature_order,/Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Pro...
1,scaler,/Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Pro...
2,ridge,/Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Pro...
3,logreg,/Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Pro...
4,linear_svc,/Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Pro...
5,poly2_logreg,/Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Pro...
6,poly2_transformer,/Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Pro...
7,baseline_comparison_summary,/Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Pro...
8,he_features,/Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Pro...
9,he_meta,/Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Pro...


### Optional TenSEAL Setup

If `tenseal` is not installed in this environment, the HE cells will skip encryption and print guidance instead of failing.

Typical setup:

- `pip install tenseal`
- make sure the active notebook kernel points to the same environment where `tenseal` is installed


In [3]:
if not TENSEAL_AVAILABLE:
    print("TenSEAL is not installed in this environment.")
    print("Install it in the notebook kernel environment, then rerun the HE cells.")
    print("Example: pip install tenseal")
else:
    print("TenSEAL is available. CKKS demo cells are ready to run.")


TenSEAL is available. CKKS demo cells are ready to run.


## 3. Load Exported Baseline Artifacts

We load the base 6D feature order, scaler, model artifacts, HE evaluation subset, and plaintext reference scores.


In [4]:
with FEATURE_ORDER_PATH.open() as f:
    feature_order = json.load(f)

with BASELINE_COMPARISON_PATH.open() as f:
    baseline_comparison_summary = json.load(f)

with SCALER_PATH.open("rb") as f:
    scaler = pickle.load(f)

with RIDGE_PATH.open("rb") as f:
    ridge_artifact = pickle.load(f)

with LOGREG_PATH.open("rb") as f:
    logreg_artifact = pickle.load(f)

with LINEAR_SVC_PATH.open("rb") as f:
    linear_svc_model = pickle.load(f)

with POLY2_LOGREG_PATH.open("rb") as f:
    poly2_logreg_model = pickle.load(f)

with POLY2_TRANSFORMER_PATH.open("rb") as f:
    poly2_transformer = pickle.load(f)

he_features_df = pd.read_csv(HE_FEATURES_PATH)
he_meta_df = pd.read_csv(HE_META_PATH)
he_scores_df = pd.read_csv(HE_SCORES_PATH)

feature_columns = feature_order["feature_columns"]
X_he = he_features_df[feature_columns].to_numpy(dtype=float)
X_he_poly = poly2_transformer.transform(X_he)

ridge_weights = np.asarray(ridge_artifact["weights"], dtype=float)
ridge_bias = float(ridge_artifact["bias"])
logreg_weights = np.asarray(logreg_artifact["weights"], dtype=float)
logreg_bias = float(logreg_artifact["bias"])
linear_svc_weights = linear_svc_model.coef_.reshape(-1).astype(float)
linear_svc_bias = float(linear_svc_model.intercept_[0])
poly2_weights = poly2_logreg_model.coef_.reshape(-1).astype(float)
poly2_bias = float(poly2_logreg_model.intercept_[0])

print("Base feature columns:", feature_columns)
print("Base HE subset shape:", X_he.shape)
print("Polynomial-expanded HE subset shape:", X_he_poly.shape)

display(pd.DataFrame(baseline_comparison_summary["models"]))
display(he_features_df.head())
display(he_scores_df.head())


/opt/anaconda3/envs/prompt_engineering/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Base feature columns: ['amount_scaled', 'oldbalanceOrg_scaled', 'oldbalanceDest_scaled', 'deltaOrig_scaled', 'deltaDest_scaled', 'is_transfer']
Base HE subset shape: (128, 6)
Polynomial-expanded HE subset shape: (128, 27)


/opt/anaconda3/envs/prompt_engineering/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/prompt_engineering/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/prompt_engineering/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator PolynomialFea

,model_name,feature_representation,score_type,he_ready,he_strategy,coefficient_shape,intercept_shape,base_feature_order,expanded_feature_dimension
0,ridge_score,base_6d_scaled,decision_function,True,encrypt base 6D scaled features and evaluate a...,[6],[],NaN,NaN
1,logistic_regression,base_6d_scaled,raw_logit,True,encrypt base 6D scaled features and evaluate a...,[6],[],NaN,NaN
2,linear_svc,base_6d_scaled,decision_function,True,encrypt base 6D scaled features and evaluate a...,[6],[1],NaN,NaN
3,poly2_logistic_regression,poly2_expanded_from_base_6d,raw_linear_score_on_poly2_features,True,pre-expand plaintext base 6D features with Pol...,[27],[1],"[amount_scaled, oldbalanceOrg_scaled, oldbalan...",27.000000


,amount_scaled,oldbalanceOrg_scaled,oldbalanceDest_scaled,deltaOrig_scaled,deltaDest_scaled,is_transfer
0,-0.307677,-0.038166,-0.394634,-0.106712,-0.240944,0
1,-0.255393,0.088375,-0.306533,-0.035529,-0.305849,0
2,-0.186655,-0.120996,-0.348592,-0.130033,-0.135959,0
3,-0.346791,-0.164399,-0.363964,-0.189789,-0.274874,0
4,-0.112125,-0.162494,-0.247061,-0.187166,-0.071306,0


,sample_row_id,isFraud,ridge_raw_score,logistic_raw_logit,logistic_probability,linear_svc_decision,poly2_logreg_raw_score,poly2_logreg_probability
0,54179,0,-0.880866,-2.408487,0.082528,-0.746430,-2.790298,0.057851
1,21497,0,-0.870142,-1.187365,0.233731,-0.428714,-2.472635,0.077799
2,12722,0,-0.895706,-4.592093,0.010030,-1.066582,-4.722244,0.008817
3,7086,0,-0.901306,-3.210373,0.038777,-0.958638,-3.264666,0.036803
4,47163,0,-0.925466,-6.861991,0.001046,-1.424033,-7.272685,0.000694


## 4. Plaintext Consistency Check

Before doing encrypted inference, we verify that saved weights and intercepts reproduce the plaintext reference scores exactly or up to machine precision.


In [5]:
model_specs = [
    {
        "model_name": "ridge_score",
        "score_label": "ridge_raw_score",
        "representation": "base_6d_scaled",
        "input_dimension": int(X_he.shape[1]),
        "X": X_he,
        "weights": ridge_weights,
        "bias": ridge_bias,
    },
    {
        "model_name": "logistic_regression",
        "score_label": "logistic_raw_logit",
        "representation": "base_6d_scaled",
        "input_dimension": int(X_he.shape[1]),
        "X": X_he,
        "weights": logreg_weights,
        "bias": logreg_bias,
    },
    {
        "model_name": "linear_svc",
        "score_label": "linear_svc_decision",
        "representation": "base_6d_scaled",
        "input_dimension": int(X_he.shape[1]),
        "X": X_he,
        "weights": linear_svc_weights,
        "bias": linear_svc_bias,
    },
    {
        "model_name": "poly2_logistic_regression",
        "score_label": "poly2_logreg_raw_score",
        "representation": "poly2_expanded_from_base_6d",
        "input_dimension": int(X_he_poly.shape[1]),
        "X": X_he_poly,
        "weights": poly2_weights,
        "bias": poly2_bias,
    },
]

plaintext_consistency_rows = []
plaintext_predictions = {}

for spec in model_specs:
    recomputed_score = spec["X"] @ spec["weights"] + spec["bias"]
    reference_score = he_scores_df[spec["score_label"]].to_numpy(dtype=float)
    abs_diff = np.abs(recomputed_score - reference_score)
    plaintext_predictions[spec["model_name"]] = recomputed_score
    plaintext_consistency_rows.append(
        {
            "model_name": spec["model_name"],
            "score_label": spec["score_label"],
            "representation": spec["representation"],
            "input_dimension": spec["input_dimension"],
            "max_abs_diff": float(abs_diff.max()),
            "mean_abs_diff": float(abs_diff.mean()),
        }
    )

plaintext_consistency_df = pd.DataFrame(plaintext_consistency_rows)
display(plaintext_consistency_df)


,model_name,score_label,representation,input_dimension,max_abs_diff,mean_abs_diff
0,ridge_score,ridge_raw_score,base_6d_scaled,6,0.000000,0.000000
1,logistic_regression,logistic_raw_logit,base_6d_scaled,6,0.000000,0.000000
2,linear_svc,linear_svc_decision,base_6d_scaled,6,0.000000,0.000000
3,poly2_logistic_regression,poly2_logreg_raw_score,poly2_expanded_from_base_6d,27,0.000000,0.000000


## 5. Single-Example Encrypted Inference Demo

We begin with one sample and compare plaintext and decrypted encrypted scores for all four models.

Interpretation note:

- Ridge, Logistic Regression, and LinearSVC all reduce to weighted sums on the same 6D input, so their HE runtime should be very similar
- The polynomial model currently uses **pre-expanded plaintext polynomial features**, so its extra HE cost comes from the higher-dimensional encrypted input rather than encrypted-space polynomial term construction


In [6]:
PRIMARY_HE_CONFIG = CKKS_PARAMETER_OPTIONS[0]

example_idx = 0

if not TENSEAL_AVAILABLE:
    single_example_df = pd.DataFrame(
        [
            {
                "status": "skipped",
                "reason": "TenSEAL is not installed in this environment.",
            }
        ]
    )
else:
    import math, time
    context = build_ckks_context(PRIMARY_HE_CONFIG)
    single_example_rows = []
    for spec in model_specs:
        example_x = spec["X"][example_idx]
        plaintext_score = float(plaintext_predictions[spec["model_name"]][example_idx])

        t0 = time.perf_counter()
        enc_vec, enc_score = encrypted_linear_score(context, example_x, spec["weights"], spec["bias"])
        encrypt_and_infer_time = time.perf_counter() - t0

        t1 = time.perf_counter()
        decrypted_score = float(enc_score.decrypt()[0])
        decrypt_time = time.perf_counter() - t1

        single_example_rows.append(
            {
                "model_name": spec["model_name"],
                "representation": spec["representation"],
                "input_dimension": spec["input_dimension"],
                "config": PRIMARY_HE_CONFIG["name"],
                "plaintext_score": plaintext_score,
                "decrypted_score": decrypted_score,
                "absolute_error": abs(plaintext_score - decrypted_score),
                "encrypt_plus_infer_time_sec": encrypt_and_infer_time,
                "decrypt_time_sec": decrypt_time,
                "ciphertext_size_bytes": len(enc_score.serialize()) if hasattr(enc_score, "serialize") else math.nan,
            }
        )

    single_example_df = pd.DataFrame(single_example_rows)

display(single_example_df)

,model_name,representation,input_dimension,config,plaintext_score,decrypted_score,absolute_error,encrypt_plus_infer_time_sec,decrypt_time_sec,ciphertext_size_bytes
0,ridge_score,base_6d_scaled,6,fast_8192,-0.880866,-0.880865,0.000001,0.009084,0.000581,235325
1,logistic_regression,base_6d_scaled,6,fast_8192,-2.408487,-2.408485,0.000001,0.008488,0.000535,235393
2,linear_svc,base_6d_scaled,6,fast_8192,-0.746430,-0.746429,0.000001,0.007694,0.000504,235348
3,poly2_logistic_regression,poly2_expanded_from_base_6d,27,fast_8192,-2.790298,-2.790297,0.000001,0.014555,0.000548,235383


## 6. Batch Encrypted Inference On The HE Evaluation Subset

We run encrypted inference one sample at a time across the HE subset and record:

- plaintext score
- decrypted score
- absolute error
- timing
- ciphertext size


In [7]:
if not TENSEAL_AVAILABLE:
    batch_comparison_df = pd.DataFrame(
        [
            {
                "status": "skipped",
                "reason": "TenSEAL is not installed in this environment.",
            }
        ]
    )
    batch_results_by_model = {}
else:
    batch_rows = []
    batch_results_by_model = {}
    for spec in model_specs:
        results_df, timing = run_ckks_batch_demo(
            spec["X"],
            plaintext_predictions[spec["model_name"]],
            spec["weights"],
            spec["bias"],
            PRIMARY_HE_CONFIG,
        )
        batch_results_by_model[spec["model_name"]] = results_df
        batch_rows.append(
            {
                "model_name": spec["model_name"],
                "representation": spec["representation"],
                "input_dimension": spec["input_dimension"],
                "config": PRIMARY_HE_CONFIG["name"],
                **timing,
            }
        )

    batch_comparison_df = pd.DataFrame(batch_rows).sort_values("model_name").reset_index(drop=True)

display(batch_comparison_df)

,model_name,representation,input_dimension,config,encrypt_time_sec,inference_time_sec,decrypt_time_sec,average_abs_error,max_abs_error,ciphertext_size_bytes,total_runtime_sec
0,linear_svc,base_6d_scaled,6,fast_8192,0.360587,0.578570,0.062239,0.000002,0.000004,"235,357.953125",1.001396
1,logistic_regression,base_6d_scaled,6,fast_8192,0.362279,0.581572,0.063767,0.000002,0.000007,"235,359.429688",1.007618
2,poly2_logistic_regression,poly2_expanded_from_base_6d,27,fast_8192,0.362701,1.405143,0.067356,0.000003,0.000010,"235,357.656250",1.835200
3,ridge_score,base_6d_scaled,6,fast_8192,0.365663,0.582717,0.062736,0.000001,0.000002,"235,356.710938",1.011116


## 7. Metrics And Timing

The table below highlights the HE comparison metrics for all four models under the primary CKKS setting.


In [8]:
if not TENSEAL_AVAILABLE:
    timing_summary_df = batch_comparison_df.copy()
else:
    timing_summary_df = batch_comparison_df[
        [
            "model_name",
            "representation",
            "input_dimension",
            "config",
            "encrypt_time_sec",
            "inference_time_sec",
            "decrypt_time_sec",
            "average_abs_error",
            "max_abs_error",
            "ciphertext_size_bytes",
            "total_runtime_sec",
        ]
    ].copy()

display(timing_summary_df)


,model_name,representation,input_dimension,config,encrypt_time_sec,inference_time_sec,decrypt_time_sec,average_abs_error,max_abs_error,ciphertext_size_bytes,total_runtime_sec
0,linear_svc,base_6d_scaled,6,fast_8192,0.360587,0.578570,0.062239,0.000002,0.000004,"235,357.953125",1.001396
1,logistic_regression,base_6d_scaled,6,fast_8192,0.362279,0.581572,0.063767,0.000002,0.000007,"235,359.429688",1.007618
2,poly2_logistic_regression,poly2_expanded_from_base_6d,27,fast_8192,0.362701,1.405143,0.067356,0.000003,0.000010,"235,357.656250",1.835200
3,ridge_score,base_6d_scaled,6,fast_8192,0.365663,0.582717,0.062736,0.000001,0.000002,"235,356.710938",1.011116


## 8. CKKS Parameter Sweep

We try a small set of CKKS settings and compare runtime, ciphertext size, and numerical error across all four models.


In [9]:
if not TENSEAL_AVAILABLE:
    sweep_df = pd.DataFrame(
        [
            {
                "status": "skipped",
                "reason": "Install TenSEAL to run the CKKS sweep.",
            }
        ]
    )
    best_by_model_df = sweep_df.copy()
else:
    sweep_rows = []
    for config in CKKS_PARAMETER_OPTIONS:
        for spec in model_specs:
            _, timing = run_ckks_batch_demo(
                spec["X"],
                plaintext_predictions[spec["model_name"]],
                spec["weights"],
                spec["bias"],
                config,
            )
            sweep_rows.append(
                {
                    "model_name": spec["model_name"],
                    "representation": spec["representation"],
                    "input_dimension": spec["input_dimension"],
                    "config": config["name"],
                    "poly_modulus_degree": config["poly_modulus_degree"],
                    "coeff_mod_bit_sizes": str(config["coeff_mod_bit_sizes"]),
                    "global_scale_bits": config["global_scale_bits"],
                    **timing,
                }
            )

    sweep_df = pd.DataFrame(sweep_rows).sort_values(
        by=["model_name", "average_abs_error", "total_runtime_sec"],
        ascending=[True, True, True],
    ).reset_index(drop=True)

    best_by_model_df = (
        sweep_df.sort_values(
            by=["model_name", "average_abs_error", "total_runtime_sec"],
            ascending=[True, True, True],
        )
        .groupby("model_name", as_index=False)
        .first()
    )

display(sweep_df)
display(best_by_model_df)


,model_name,representation,input_dimension,config,poly_modulus_degree,coeff_mod_bit_sizes,global_scale_bits,encrypt_time_sec,inference_time_sec,decrypt_time_sec,average_abs_error,max_abs_error,ciphertext_size_bytes,total_runtime_sec
0,linear_svc,base_6d_scaled,6,fast_8192,8192,"[60, 40, 40, 60]",40,0.361874,0.581183,0.062568,0.000001,0.000003,"235,357.312500",1.005625
1,linear_svc,base_6d_scaled,6,precise_16384,16384,"[60, 40, 40, 40, 40, 60]",40,1.033928,2.910609,0.299123,0.000002,0.000024,"855,654.765625",4.243661
2,linear_svc,base_6d_scaled,6,balanced_16384,16384,"[60, 40, 40, 40, 60]",40,0.897429,2.009468,0.211167,0.000007,0.000031,"657,764.906250",3.118064
3,logistic_regression,base_6d_scaled,6,fast_8192,8192,"[60, 40, 40, 60]",40,0.370833,0.593779,0.065498,0.000002,0.000007,"235,355.695312",1.030110
4,logistic_regression,base_6d_scaled,6,precise_16384,16384,"[60, 40, 40, 40, 40, 60]",40,1.031421,2.900361,0.301145,0.000009,0.000090,"855,596.234375",4.232927
5,logistic_regression,base_6d_scaled,6,balanced_16384,16384,"[60, 40, 40, 40, 60]",40,0.893029,2.003687,0.212932,0.000011,0.000084,"657,766.054688",3.109648
6,poly2_logistic_regression,poly2_expanded_from_base_6d,27,fast_8192,8192,"[60, 40, 40, 60]",40,0.367104,1.414307,0.068612,0.000002,0.000014,"235,355.726562",1.850022
7,poly2_logistic_regression,poly2_expanded_from_base_6d,27,precise_16384,16384,"[60, 40, 40, 40, 40, 60]",40,1.029955,7.277681,0.313210,0.000011,0.000130,"855,676.070312",8.620846
8,poly2_logistic_regression,poly2_expanded_from_base_6d,27,balanced_16384,16384,"[60, 40, 40, 40, 60]",40,0.895587,4.950070,0.216268,0.000012,0.000127,"657,848.156250",6.061924
9,ridge_score,base_6d_scaled,6,fast_8192,8192,"[60, 40, 40, 60]",40,0.362919,0.582624,0.061955,0.000000,0.000001,"235,366.335938",1.007499


,model_name,representation,input_dimension,config,poly_modulus_degree,coeff_mod_bit_sizes,global_scale_bits,encrypt_time_sec,inference_time_sec,decrypt_time_sec,average_abs_error,max_abs_error,ciphertext_size_bytes,total_runtime_sec
0,linear_svc,base_6d_scaled,6,fast_8192,8192,"[60, 40, 40, 60]",40,0.361874,0.581183,0.062568,0.000001,0.000003,"235,357.312500",1.005625
1,logistic_regression,base_6d_scaled,6,fast_8192,8192,"[60, 40, 40, 60]",40,0.370833,0.593779,0.065498,0.000002,0.000007,"235,355.695312",1.030110
2,poly2_logistic_regression,poly2_expanded_from_base_6d,27,fast_8192,8192,"[60, 40, 40, 60]",40,0.367104,1.414307,0.068612,0.000002,0.000014,"235,355.726562",1.850022
3,ridge_score,base_6d_scaled,6,fast_8192,8192,"[60, 40, 40, 60]",40,0.362919,0.582624,0.061955,0.000000,0.000001,"235,366.335938",1.007499


## 9. Final Summary

The final summary identifies the current best tradeoff and gives the main interpretation for teammates:

- linear models on the same 6D base input should have very similar HE cost
- the polynomial model adds HE overhead because it encrypts a larger pre-expanded feature vector


In [10]:
if not TENSEAL_AVAILABLE:
    print("Final summary")
    print("- Plaintext consistency checks passed using saved model weights and intercepts.")
    print("- TenSEAL is not installed here, so encrypted CKKS runs were skipped.")
    print("- After installing TenSEAL, rerun the HE cells to populate the comparison tables.")
else:
    overall_best_row = sweep_df.sort_values(
        by=["average_abs_error", "total_runtime_sec"],
        ascending=[True, True],
    ).iloc[0].to_dict()

    print("Final summary")
    print("- Primary encrypted target for the base logistic model remains the raw logit.")
    print("- Linear models (ridge, logistic regression, LinearSVC) all use weighted sums on the same 6D encrypted input, so their HE costs should stay in the same range.")
    print("- The polynomial model uses pre-expanded plaintext degree-2 features, so its extra HE cost comes from encrypting and scoring a larger feature vector instead of constructing polynomial terms in encrypted space.")
    print("- Best overall CKKS setting in this run:", overall_best_row["config"])
    print("- Best-by-model settings:")
    for row in best_by_model_df.to_dict(orient="records"):
        print(
            f"  - {row['model_name']}: config={row['config']}, "
            f"avg_abs_error={row['average_abs_error']:.6e}, "
            f"max_abs_error={row['max_abs_error']:.6e}, "
            f"total_runtime_sec={row['total_runtime_sec']:.6f}"
        )
    print("- Verdict: encrypted inference tracks plaintext closely enough for the prototype demo.")


Final summary
- Primary encrypted target for the base logistic model remains the raw logit.
- Linear models (ridge, logistic regression, LinearSVC) all use weighted sums on the same 6D encrypted input, so their HE costs should stay in the same range.
- The polynomial model uses pre-expanded plaintext degree-2 features, so its extra HE cost comes from encrypting and scoring a larger feature vector instead of constructing polynomial terms in encrypted space.
- Best overall CKKS setting in this run: fast_8192
- Best-by-model settings:
  - linear_svc: config=fast_8192, avg_abs_error=8.737962e-07, max_abs_error=3.073078e-06, total_runtime_sec=1.005625
  - logistic_regression: config=fast_8192, avg_abs_error=2.359183e-06, max_abs_error=6.773996e-06, total_runtime_sec=1.030110
  - poly2_logistic_regression: config=fast_8192, avg_abs_error=1.819285e-06, max_abs_error=1.358162e-05, total_runtime_sec=1.850022
  - ridge_score: config=fast_8192, avg_abs_error=2.729047e-07, max_abs_error=5.884823e-

## 10. Export HE Timing Results

Save the CKKS sweep timing data to `artifacts/he_timing_results.json` for use in `visualization.ipynb`.


In [11]:
import json as _json

if TENSEAL_AVAILABLE and 'sweep_df' in dir() and not sweep_df.empty and 'status' not in sweep_df.columns:
    he_timing_path = ARTIFACT_DIR / "he_timing_results.json"
    sweep_records = sweep_df[[
        "model_name", "representation", "input_dimension", "config",
        "poly_modulus_degree", "encrypt_time_sec", "inference_time_sec",
        "decrypt_time_sec", "average_abs_error", "max_abs_error",
        "ciphertext_size_bytes", "total_runtime_sec",
    ]].to_dict(orient="records")
    with he_timing_path.open("w") as _f:
        _json.dump(sweep_records, _f, indent=2)
    print("Saved he_timing_results.json")
    display(sweep_df[["model_name", "config", "total_runtime_sec", "average_abs_error"]])
else:
    print("TenSEAL not available or sweep not run — he_timing_results.json not saved.")

Saved he_timing_results.json


,model_name,config,total_runtime_sec,average_abs_error
0,linear_svc,fast_8192,1.005625,0.000001
1,linear_svc,precise_16384,4.243661,0.000002
2,linear_svc,balanced_16384,3.118064,0.000007
3,logistic_regression,fast_8192,1.030110,0.000002
4,logistic_regression,precise_16384,4.232927,0.000009
5,logistic_regression,balanced_16384,3.109648,0.000011
6,poly2_logistic_regression,fast_8192,1.850022,0.000002
7,poly2_logistic_regression,precise_16384,8.620846,0.000011
8,poly2_logistic_regression,balanced_16384,6.061924,0.000012
9,ridge_score,fast_8192,1.007499,0.000000
